# Analysis Overview

Central entry point for all analysis notebooks.
Core questions, notebook index and key findings summary.

## Core Questions

1. Where do delays occur? — stops, districts, lines
2. When do delays occur? — time of day, weekday, season
3. What amplifies delays? — weather, events
4. What does the target itself look like? — distribution, OTP, arr vs dep
5. Which features correlate most with delays?
6. Can delays be predicted? → modeling

## Notebooks

| Notebook | Focus |
|:---|:---|
| `03_analysis_target.ipynb` | Delay distribution, OTP, arr vs dep, cancellations |
| `03_analysis_network.ipynb` | Netzveränderungen 2023–2025 · Vor/Nachher · Einlaufzeit · Hotspots · Versorgungsqualität |
| `03_analysis_temporal.ipynb` | Hour · weekday · month · season · full year |
| `03_analysis_spatial.ipynb` | Stops · districts · lines |
| `03_analysis_weather.ipynb` | Rain · wind · snow · temperature |
| `03_analysis_events.ipynb` | Holidays · events · event size |

> **Hinweis für alle Notebooks:** Das Tramnetz hat sich im Analysezeitraum 2023–2025 verändert.
> Fahrplanwechsel Dezember 2023 (j23 → j24): Linien 9, 11 und 13 wurden fundamental umgebaut.
> Bei linienbezogenen Befunden immer `03_analysis_network.ipynb` als Kontext heranziehen.

## Line Colors

Offizielle VBZ-Linienfarben aus GTFS `routes.txt` — verfügbar via `line_color("12")` aus `zh_tram_flow.config`.

| Linie | Farbe | | Linie | Farbe | | Linie | Farbe |
|:---:|:---|:---|:---:|:---|:---|:---:|:---|
| **2** | 🟥 `#E20A16` | | **8** | 🟩 `#8AB51F` | | **14** | 🟦 `#008DC5` |
| **3** | 🟩 `#00892F` | | **9** | 🟦 `#11296F` | | **15** | 🟥 `#E20A16` |
| **4** | 🟦 `#11296F` | | **10** | 🟪 `#E12472` | | **17** | 🟥 `#8E224D` |
| **5** | 🟫 `#734522` | | **11** | 🟩 `#00892F` | | **19** | 🟥 `#E20A16` |
| **6** | 🟧 `#CA7D3C` | | **12** | 🩵 `#92D6E3` | | **E** | 🟥 `#E20A16` |
| **7** | ⬛ `#000000` | | **13** | 🟨 `#FFCC00` | | | |

## Key Findings

> Alle Findings werden in den jeweiligen Analysis-Notebooks erarbeitet und hier zentral gelistet.
> ID-Schema: `F-{NOTEBOOK}-{NR}` — Status: `open` · `in-progress` · `done` · `⚠️ aktiv`

| ID | Notebook | Section | Finding | Impact | Action | Action Location | Status |
|:---|:---|:---|:---|:---|:---|:---|:---|
| F-TARGET-01 | target | Delay Distribution | `arrival_delay` rechtsschiefe Verteilung — Median ≠ Mean | Lineare Modelle unterschätzen Extremwerte | Log-Transform + MdAE-Robustness-Check | `03_analysis_target` — Log Transform | open |
| F-TARGET-02 | target | Delay Delta Distribution | `delay_delta` bimodal — Terminus-Cluster bei −50s | Systematisches Signal, kein Fehler | Terminus-Flag als Feature ableiten + räumlich verorten | `03_analysis_spatial` | open |
| F-TARGET-03 | target | On-Time Performance | 70% der Halte haben `delay_delta > 0` — kein ausreichender Puffer | Fahrplan zu knapp kalkuliert | Puffer-Feature aus Dwell-Time ableiten | `02_preparation` | open |
| F-TARGET-04 | target | Target Definition | `dep_schedule − arr_schedule` = Dwell-Time-Feature verfügbar | Wichtiges Feature für Verspätungsrisiko pro Halt | Als `dwell_time` in Preparation aufnehmen | `02_preparation` | open |
| F-TARGET-05 | target | Cancellations by Line | `canceled`-Flag netzweit erhöht Jan 2023 – Jun 2024, simultane Normalisierung Jul 2024 — wahrscheinlich Datendefinitions-Änderung, kein Infrastrukturproblem | Verfälscht Cancellation-Baseline im gesamten Training-Set | (1) `canceled=True` aus Delay-Modell ausschließen; (2) `is_pre_july_2024` als Feature für Cancellation-Modell | `02_preparation` | open |
| F-TARGET-06 | target | Monthly Trend | Nov–Dez 2025 Fahrplanwechsel-Artefakt (j25→j26) — delta +17s/+26s | Artificial Noise im Training- und Test-Set | Nov–Dez 2025 aus Train+Test entfernen (Strategie A) | `02_preparation` | ⚠️ aktiv |
| F-TARGET-07 | target | Extreme Values | Extremwerte bis +5000s vorhanden | Potenzielle Messfehler oder echte Störungsereignisse | Abgleich mit Wetter- und Ereignis-Daten | `03_analysis_temporal` | open |
| F-TARGET-08 | target | On-Time Performance | `trip_id` fehlt — Kaskadenwirkungen zwischen Fahrten nicht sichtbar | Wichtiges Prediction-Signal fehlt | `trip_id` in Pipeline re-integrieren (Task läuft) | `02_preparation` (sf_data-research) | open |
| F-TARGET-09 | target | Delay Overview Per Year | Langfristiger Delta-Aufwärtstrend +4.4s (2023) → +7.7s (2025) | Strukturelles Problem, kein Einmaleffekt | Jahr + Monat als Features in Modell aufnehmen | Modell-Phase | open |

## Setup

In [ ]:
from zh_tram_flow.notebook import *

TRAIN, TEST, lf = setup_analysis("03_analysis")

%load_ext autoreload
%autoreload 2